# Projeto (Seção 7) — solução comentada

Continua a solução comentada da seção 6. Enunciado em [`05_projeto`](05_projeto.ipynb).

## 1. Tipos reutilizáveis (testados isoladamente, como o enunciado sugere)

In [1]:
from datetime import date
from enum import Enum
from typing import Annotated, TypeVar
from uuid import UUID, uuid4
from pydantic import UUID4, BaseModel, ConfigDict, Field, TypeAdapter, ValidationError, field_serializer
from pydantic.alias_generators import to_camel

T = TypeVar("T")
BoundedString = Annotated[str, Field(min_length=2, max_length=50)]
BoundedList = Annotated[list[T], Field(min_length=1, max_length=5)]

Em vez de criar modelos de teste de um campo, `TypeAdapter` valida **o tipo diretamente** — mais enxuto para testar tipos anotados:

In [2]:
def check(tp, value) -> str:
    try:
        return f"ok -> {TypeAdapter(tp).validate_python(value)!r}"
    except ValidationError as ex:
        return " | ".join(f"{e['loc']} {e['type']}" for e in ex.errors())

cases = [
    ("BoundedString", BoundedString, "ab"),
    ("BoundedString", BoundedString, "a"),
    ("BoundedString", BoundedString, "x" * 50),
    ("BoundedString", BoundedString, "x" * 51),
    ("BoundedList[int]", BoundedList[int], []),
    ("BoundedList[int]", BoundedList[int], [1, 2, 3, 4, 5]),
    ("BoundedList[int]", BoundedList[int], [1] * 6),
    ("BoundedList[BoundedString]", BoundedList[BoundedString], ["aa", "b", "c" * 60]),
]
for label, tp, value in cases:
    shown = value if len(str(value)) <= 20 else f"<{len(value)} itens/caracteres>"
    print(f"{label}({shown!r}): {check(tp, value)}")

BoundedString('ab'): ok -> 'ab'
BoundedString('a'): () string_too_short
BoundedString('<50 itens/caracteres>'): ok -> 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'
BoundedString('<51 itens/caracteres>'): () string_too_long
BoundedList[int]([]): () too_short
BoundedList[int]([1, 2, 3, 4, 5]): ok -> [1, 2, 3, 4, 5]
BoundedList[int]([1, 1, 1, 1, 1, 1]): () too_long
BoundedList[BoundedString]('<3 itens/caracteres>'): (1,) string_too_short | (2,) string_too_long


## 2. Modelo

In [3]:
class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


def make_alias(field_name: str) -> str:
    return to_camel(field_name).removesuffix("_")


class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=make_alias,
    )

    id_: UUID4 = Field(default_factory=uuid4)               # continua NÃO anulável (requisito da seção 6)
    manufacturer: BoundedString
    series_name: BoundedString
    type_: AutomobileType
    is_electric: bool = False
    manufactured_date: date = Field(validation_alias="completionDate", ge=date(1980, 1, 1))
    base_msrp_usd: float = Field(validation_alias="msrpUSD", serialization_alias="baseMSRPUSD")
    top_features: BoundedList[BoundedString] | None = None  # alias topFeatures gerado
    vin: BoundedString
    number_of_doors: int = Field(default=4, validation_alias="doors", ge=2, le=4, multiple_of=2)
    registration_country: BoundedString | None = None
    license_plate: BoundedString | None = None

    @field_serializer("manufactured_date", when_used="json-unless-none")
    def serialize_date(self, value: date) -> str:
        return value.strftime("%Y/%m/%d")

## 3. Testes do enunciado e extras

In [4]:
data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4 Competition xDrive",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "topFeatures": ["6 cylinders", "all-wheel drive", "convertible"],
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "France",
    "licensePlate": "AAA-BBB"
}

expected_serialized_by_alias = {
    'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'),
    'manufacturer': 'BMW',
    'seriesName': 'M4 Competition xDrive',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'topFeatures': ['6 cylinders', 'all-wheel drive', 'convertible'],
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'France',
    'licensePlate': 'AAA-BBB'
}

In [5]:
car = Automobile.model_validate(data)
assert car.model_dump(by_alias=True) == expected_serialized_by_alias
fields = list(Automobile.model_fields)
assert fields.index("top_features") == fields.index("vin") - 1          # logo antes de vin
print("teste do enunciado: ok")

def errs(**changes) -> list[tuple]:
    try:
        Automobile.model_validate({**data, **changes})
    except ValidationError as ex:
        return [(err["loc"], err["type"]) for err in ex.errors()]
    return []

print(f"topFeatures=[]: {errs(topFeatures=[])}")
print(f"topFeatures com 6 itens: {errs(topFeatures=['ab'] * 6)}")
print(f"topFeatures=['ok', 'x']: {errs(topFeatures=['ok', 'x'])}")
print(f"topFeatures=None: {errs(topFeatures=None)}")
print(f"manufacturer='B': {errs(manufacturer='B')}")
print(f"licensePlate=None: {errs(licensePlate=None)}")
print(f"id=None: {errs(id=None)}")

# interação com str_strip_whitespace: o limite mínimo é medido DEPOIS do strip
print(f"manufacturer='  B  ': {errs(manufacturer='  B  ')}")
print("testes extras: ok")

teste do enunciado: ok
topFeatures=[]: [(('topFeatures',), 'too_short')]
topFeatures com 6 itens: [(('topFeatures',), 'too_long')]
topFeatures=['ok', 'x']: [(('topFeatures', 1), 'string_too_short')]
topFeatures=None: []
manufacturer='B': [(('manufacturer',), 'string_too_short')]
licensePlate=None: []
id=None: [(('id',), 'uuid_type')]
manufacturer='  B  ': [(('manufacturer',), 'string_too_short')]
testes extras: ok


## 4. Comparação com a solução do instrutor

| ponto | instrutor | esta solução |
|---|---|---|
| tipos `BoundedString` / `BoundedList[T]` | iguais | iguais |
| `id_` | `UUID4 \| None` ⚠️ regressão do requisito da seção 6 | `UUID4` (não anulável) |
| testes dos tipos | modelos `Test` de um campo | `TypeAdapter` direto no tipo |
| testes do modelo | caminho feliz | + posição do campo, lista vazia/longa, item curto, `None`, strip × min_length |

O teste do `manufacturer='  B  '` mostra uma interação útil: `str_strip_whitespace` (do `model_config`) roda **antes** do `min_length`, então
`"  B  "` (5 caracteres) é rejeitado por ter 1 caractere útil — exatamente o comportamento que queremos.